# And

Bloqs for doing "AND" logical operations.

The behavior is modified by the 'control variable' attributes. A traditional value of '1'
means that a bit value of '1' is logical true for the and operation. A control value of
'0' means that a bit value of '0' is the logical true.

The `Toffoli` bloq is similar to the `And` bloq. Toffoli will flip the target bit according
to the and of its control registers. `And` will output the result into a fresh register.

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0. The call graphs show declared calls. The compute and erase diagrams show FlagQuantum instructions, including the measurement, feedback and reset.

In [ ]:
import numpy as np
from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg


def show_calls(graph):
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

## `And`
A two-bit 'and' operation optimized for T-count.

#### Parameters
 - `cv1`: Whether the first bit is a positive control.
 - `cv2`: Whether the second bit is a positive control. 

#### Registers
 - `ctrl`: A two-bit control register.
 - `target [right]`: The output bit. 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662).     Babbush et al. 2018. Section III.A. and Fig. 4.
 - [Verifying Measurement Based Uncomputation](https://algassert.com/post/1903). Gidney, C. 2019.


In [ ]:
from examples.algorithms.and_notebook import AndLadder, and_signature, and_source_graph
from flagquantum.dynamic import DynamicCircuit

### Example Instances

In [ ]:
and_values = (1, 1)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("And", and_signature(2, multi=False), {})])))

### Call Graph

In [ ]:
show_calls(and_source_graph(and_values))

### Clifford + T implementation

`And` will be considered an Atomic (non-decomposable) bloq within the Qualtran standard library. An additional method is provided to assist in fully compiling to a Clifford+T gateset.

In [ ]:
compute = DynamicCircuit(3).append(AndLadder(and_values))
print(compute.draw())

In [ ]:
erase = DynamicCircuit(3).append(AndLadder(and_values, erase=True))
print(erase.draw())

## `MultiAnd`
A many-bit (multi-control) 'and' operation.

#### Parameters
 - `cvs`: A tuple of control variable settings. Each entry specifies whether that control line is a "positive" control (`cv[i]=1`) or a "negative" control `0`. If a HasLength object is passed, assumes the control values to be all 1's.  

#### Registers
 - `ctrl`: An n-bit control register.
 - `junk [right]`: An `n-2` bit junk register to be cleaned up by the inverse operation.
 - `target [right]`: The output bit.


In [ ]:
from examples.algorithms.conjunction_tensors import conjunction_program

### Example Instances

In [ ]:
multi_values = (1, 0, 1, 0, 1, 0)
multi_and = conjunction_program(multi_values)

#### Graphical Signature

In [ ]:
from flagquantum.core.ir.program.signatures import Signature

multi_signature = Signature(multi_and.functions[0].registers)
display(SVG(field_signature_svg([("MultiAnd(6)", multi_signature, {})])))

### Call Graph

In [ ]:
show_calls(and_source_graph(multi_values))

## Additional Demos

### Testing with states and effects

We can use `ZeroState` and its friends to test the truth table on this classical logic.

In [ ]:
from examples.algorithms.and_notebook import projected_conjunction
from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_graph import export_register_graph_dot

cvs = (1, 1, 1)
ctrl_string = (1, 1, 1)
program = conjunction_program(cvs)
score = component_register_score(program, "multi_and")
display(SVG(render_call_graph_svg(export_register_graph_dot(score))))

In [ ]:
vec = projected_conjunction(cvs, ctrl_string)
junk_i, res_i = np.where(vec.numpy().reshape(2, 2))
display(res_i)

In [ ]:
should_be = ctrl_string == cvs
assert bool(res_i[0]) == should_be
should_be

### Classical Simulation

The `And` gate is classical logic, so we can simulate it on discrete bitstrings.

In [ ]:
from flagquantum.simulation.classical_and import and_classical_values, multi_and_classical_values

and_classical_values(np.array([1, 1], dtype=np.uint8))["target"]

In [ ]:
ctrl = np.array([1, 1, 1, 1], dtype=np.uint8)
multi_and_classical_values(ctrl, (1, 1, 1, 1))["target"]

In [ ]:
from examples.algorithms.register_graphs import classical_predicate_dot

display(SVG(render_call_graph_svg(classical_predicate_dot())))